# Cross-validation and model selection

## Motivation

- Underfitting vs. overfitting
  - Low-degree polynomials can miss the underlying relationship
  - High-degree polynomials can follow sample noise
  - A closer fit to training data need not improve new predictions
- Single holdout split (as in lecture 1):
  - A single validation split can give an unstable comparison
  - Repeated choices based on the test set weaken its independence
- Use cross-validation to select complexity and reserve the test set for final
  evaluation

***
## Optimizing hyperparameters with cross-validation

### Modeling workflow

- **Hyperparameters:** choices such as polynomial degree
- **Coefficients:** estimated from data once these choices are fixed

1. Set aside an independent test sample
2. Divide the remaining data into $K$ folds
   - Fit on $K-1$ folds and validate on the remaining fold
   - Repeat for every fold and average the validation scores
3. Choose hyperparameters and refit on all training observations
4. Evaluate the refitted model once on the test sample

![CV split](cv_split.svg)

***
### Ames data and cross-validation setup

- Actual sale prices from `ames_houses.csv`
- Target: `SalePrice`, in thousands of USD
- Predictors: `LivingArea` and `YearBuilt`

#### Data preparation

- Living area at most 350 m², lot area at most 5,000 m²
- Observations with complete selected predictors and target

In [ ]:
def load_ames_data(file, features, target):
    """
    Load filtered Ames observations for selected features and a target.

    Parameters
    ----------
    file : str
        Path to the Ames CSV file
    features : list of str
        Column names used as predictors
    target : str
        Column name used as the response

    Returns
    -------
    X : DataFrame
        Selected predictors
    y : Series
        Selected response
    """
    import pandas as pd

    df_ames = pd.read_csv(file)
    df_ames = df_ames.query('LivingArea <= 350 & LotArea <= 5000')
    df_ames = df_ames.dropna(subset=[*features, target])
    return df_ames[features], df_ames[target]

The following cell

1. Defines the target and predictors
2. Loads the Ames housing data 
3. Splits the data into training and test samples

In [ ]:
from sklearn.model_selection import train_test_split

# Relative path to the data folder in the git repository
DATA_PATH = '../../data'

# Path to Ames housing data
file = f'{DATA_PATH}/ames_houses.csv'

# Features and target variables
features = ['LivingArea', 'YearBuilt']
target = 'SalePrice'

# Load the Ames housing data
X_ames, y_ames = load_ames_data(file, features, target)

# Set aside 30% of the observations for the independent test sample
X_train, X_test, y_train, y_test = train_test_split(
    X_ames, y_ames, test_size=0.3, random_state=1234
)

print(f'Number of training observations: {len(y_train):,d}')
print(f'Number of test observations: {len(y_test):,d}')

#### Cross-validation setup

- Five folds of approximately equal size created with
  [`KFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html)
- Shuffle because houses are treated as a pooled cross-section
  - Time series require splits that respect chronological order
- `KFold` arguments: `n_splits=5`, `shuffle=True`, `random_state=1234`

In [ ]:
from sklearn.model_selection import KFold

# TODO: Create cv_splitter with five shuffled folds and random_state=1234

- Scikit-learn scores follow a "larger is better" convention
  - Maximizing negative RMSE is equivalent to minimizing RMSE
  - Negate the mean score to report positive RMSE
- List of supported scorer names: [scoring documentation](https://scikit-learn.org/stable/modules/model_evaluation.html#string-name-scorers)

In [ ]:
# TODO: Set scoring to the negative root mean squared error scoring rule

### Evaluating a quadratic model

Build a pipeline with four named steps using
  [`Pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html):
1. `input_scale`: center and scale the original inputs with
  [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)
2. `poly`: create powers and interactions with
  [`PolynomialFeatures`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html)
3. `scale`: standardize the expanded columns
4. `lr`: fit linear regression with an intercept using
  [`LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)
  
The pipeline ensures that all preprocessing happens within CV:
- The entire pipeline is fitted within each fold
- Scaling uses only that fold's training observations
- Validation observations do not determine the transformations

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

# TODO: Create pipe for degree 2 with steps input_scale, poly, scale, and lr

- Evaluate the quadratic pipeline and obtain one validation score per fold with
  [`cross_val_score()`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html)
- Arguments: 
  - `pipe`: the pipeline to evaluate
  - `X_train` and `y_train`: the training data
  - `cv`: the `KFold` splitter
  - `scoring`: the scoring metric (`neg_root_mean_squared_error`)
  - `n_jobs=4`: number of jobs to run in parallel (optional)
- A fresh copy of the entire pipeline is fitted in every fold

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score

# TODO: Compute quadratic_scores with cross_val_score()
# quadratic_scores = cross_val_score(...)

pd.DataFrame(
    {'Score': quadratic_scores},
    index=pd.Index(np.arange(len(quadratic_scores)) + 1, name='Fold'),
)

In [ ]:
# TODO: Compute quadratic_rmse from the negative validation scores

print(f'Quadratic model CV RMSE: {quadratic_rmse:.3f}')

#### Comparing training and validation errors

- `cross_val_score()` returns only validation scores
- Training and validation scores with
  [`cross_validate()`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html)
  - `return_train_score=True`: include training scores
- The returned dictionary contains:
  - `train_score`: training scores for each fold
  - `test_score`: held-out *validation* scores for each fold
  - `fit_time` and `score_time`
- Large gap between training and validation RMSE: possible overfitting

In [ ]:
from sklearn.model_selection import cross_validate

# TODO: Compute scores_quadratic, including training scores
# scores_quadratic = cross_validate(...)

In [ ]:
# TODO: Compute quadratic_train_rmse from scores_quadratic['train_score']
# TODO: Compute quadratic_validation_rmse from scores_quadratic['test_score']

print(f'Mean training RMSE: {quadratic_train_rmse:.3f}')
print(f'Mean validation RMSE: {quadratic_validation_rmse:.3f}')

### Choosing the polynomial degree

- Automatically test polynomial degrees 1 through 9 with
  [`validation_curve()`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.validation_curve.html)
- The function handles the candidate values and folds
- Need to specify the parameter name for the hyperparameter being tested
  - Single estimator: parameter name = attribute name
  - Pipeline: `step__parameter`
    - `poly__degree`: degree in the step named `poly`
- Score arrays have shape `(len(degrees), K)`
- Rows are degrees, columns are folds

In [ ]:
from sklearn.model_selection import validation_curve

# Sequence of polynomial degrees to evaluate
degrees = np.arange(1, 10)

# TODO: Compute train_scores and validation_scores with validation_curve()
# train_scores, validation_scores = validation_curve(...)

In [ ]:
# TODO: Compute rmse_train and rmse_validation by averaging across folds
# rmse_train = ...
# rmse_validation = ...

# TODO: Find index of minimum validation RMSE and corresponding degree
# imin = ...
# dmin = ...

print(f'Degree with minimum validation RMSE: {dmin}')
print(f'CV selection RMSE: {rmse_validation[imin]:.3f}')

Validation curve plot:

- Use function `plot_validation_curve()` defined below
- Mean training and validation RMSE by polynomial degree
- Vertical line: degree with minimum validation RMSE

In [ ]:
def plot_validation_curve(degrees, rmse_train, rmse_validation, dmin):
    """
    Plot mean training and validation RMSEs against polynomial degree.

    Parameters
    ----------
    degrees : array-like
        Candidate polynomial degrees
    rmse_train : array-like
        Mean training RMSE for each degree
    rmse_validation : array-like
        Mean validation RMSE for each degree
    dmin : int
        Degree with the lowest mean validation RMSE
    """
    import matplotlib.pyplot as plt

    _fig, ax = plt.subplots(figsize=(5, 3.5), layout='constrained')
    ax.plot(degrees, rmse_train, c='black', marker='x', ms=4, label='Training')
    ax.plot(
        degrees, rmse_validation, c='steelblue', marker='o', ms=4, label='Validation'
    )
    ax.axvline(dmin, ls=':', lw=0.75, c='black')
    ax.set(xlabel='Polynomial degree', ylabel='RMSE (thousand USD)', xticks=degrees)
    ax.legend()

In [ ]:
plot_validation_curve(degrees, rmse_train, rmse_validation, dmin)

### Selecting and evaluating the final model

- Hyperparameter search with
  [`GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)
  - Parameter grid: dictionary of parameter names and candidate values
  - `refit=True`: selected pipeline refitted on all training observations

Fitted `GridSearchCV` object has attributes:

- `best_params_`: selected hyperparameter values
- `best_score_`: mean CV *selection score*
- `best_estimator_`: pipeline refitted on all training data

In [ ]:
from sklearn.model_selection import GridSearchCV

# TODO: Define parameter grid for polynomial degrees
# param_grid = ...

# TODO: Create polynomial grid search with the shared CV setup and refit=True
# poly_cv = GridSearchCV(...)

# TODO: Fit poly_cv on X_train, y_train

In [ ]:
# TODO: Print poly_cv.best_params_ and the CV selection RMSE

- Predictions from the refitted best estimator
- RMSE in thousand USD with
  [`root_mean_squared_error()`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_error.html)
- $R^2$ with
  [`r2_score()`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html)

In [ ]:
from sklearn.metrics import r2_score, root_mean_squared_error

# TODO: Extract estimator from poly_cv.best_estimator_
# estimator = ...

# TODO: Predict y_test_pred for X_test
# y_test_pred = ...

# TODO: Compute test_rmse and test_r2
# test_rmse = ...
# test_r2 = ...

print(f'Test RMSE (thousand USD): {test_rmse:.3f}')
print(f'Test R²: {test_r2:.3f}')

<div style="padding: 0.8em 1em 0.5em 1em; border: 2pt solid #1d91c0;">
<p style="font-weight: bold; font-size: 1.5em; color: #1d91c0;">Your turn</p>

1. Repeat the polynomial-degree grid search using $R^2$ as the scoring criterion.
   Keep the same training observations, candidate degrees, and CV folds.
   Set `scoring='r2'`; larger scores indicate better predictions.
2. Report the selected degree and its mean validation $R^2$. Does the selected
   degree differ from the one chosen using RMSE?

</div>
<span style="display: none;">YourTurnEnd</span>

***
## Principal component analysis

### A geometric introduction to principal component analysis

- Principal component analysis (PCA) rotates the predictor axes
    - The first principal component captures the greatest variation
    - Later components capture remaining variation in perpendicular directions
- PCA is *unsupervised*: it does not use the response variable


In [ ]:
def generate_bivariate_normal(N=100, sigma1=1.5, rho=0.8):
    """
    Generate a synthetic bivariate normal dataset for PCA demonstration.

    Parameters
    ----------
    N : int
        Number of observations to generate
    sigma1 : float
        Standard deviation of the first variable
    rho : float
        Correlation coefficient between the two variables

    Returns
    -------
    X : array of shape (N, 2)
        Generated synthetic dataset with two centered variables
    """
    import numpy as np

    rng = np.random.default_rng(seed=1234)
    cov = np.array([[sigma1**2, sigma1 * rho], [sigma1 * rho, 1]])
    X = rng.multivariate_normal(mean=[0, 0], cov=cov, size=N)
    # Center the variables to have mean zero
    X = X - X.mean(axis=0)
    return X

In [ ]:
from sklearn.decomposition import PCA

# Create a bivariate normal demo dataset
X = generate_bivariate_normal(N=100, rho=0.8)

# Set up PCA with 2 components
pca = PCA(n_components=2)

# Perform PCA on the synthetic dataset
X_pc = pca.fit_transform(X)

print('Explained variance fractions:', pca.explained_variance_ratio_)

In [ ]:
def plot_pca_demo(X, X_pc, pca):
    """
    Plot the original and PCA-transformed datasets with principal axes.

    Parameters
    ----------
    X : array of shape (N, 2)
        Original dataset
    X_pc : array of shape (N, 2)
        PCA-transformed dataset
    pca : PCA object
        Fitted PCA object containing mean and components
    """
    import matplotlib.pyplot as plt
    import numpy as np

    _fig, axes = plt.subplots(
        1, 2, figsize=(6.5, 3.5), layout='constrained', sharex=True, sharey=True
    )

    # Data used for plotting the principal axes and arrows
    origins = [pca.mean_, np.zeros(2)]
    directions = [pca.components_, np.eye(2)]

    colors = ['black', 'red']
    lims = (-6, 6)

    for i, ax in enumerate(axes):
        # Plot the original or PCA-transformed data
        data = X if i == 0 else X_pc
        ax.scatter(data[:, 0], data[:, 1], s=18, c='none', ec='steelblue', lw=0.6)

        ax.set_xlabel('$x_1$' if i == 0 else 'PC1')
        ax.set_ylabel('$x_2$' if i == 0 else 'PC2')
        ax.set_title('Before PCA' if i == 0 else 'After PCA')
        ax.set(xlim=lims, ylim=lims, aspect=1)

        # Draw each principal axis and an arrow
        for j in range(2):
            # Scale the arrow length by 3 std. dev. to make it visible in the plot
            arrow = 3 * np.sqrt(pca.explained_variance_[j]) * directions[i][j]
            endpoint = origins[i] + arrow
            ax.axline(origins[i], endpoint, c=colors[j], lw=1, label=f'PC{j + 1}')
            ax.annotate(
                '',
                xy=endpoint,
                xytext=origins[i],
                arrowprops={'arrowstyle': '->', 'lw': 2, 'color': colors[j]},
            )

    axes[0].legend()

In [ ]:
plot_pca_demo(X, X_pc, pca)

<div style="padding: 0.8em 1em 0.5em 1em; border: 2pt solid #1d91c0;">
<p style="font-weight: bold; font-size: 1.5em; color: #1d91c0;">Your turn</p>

1. Use `generate_bivariate_normal()` with `rho=0`, keeping the other arguments
   unchanged. Repeat PCA and its visualization.
2. Compare the principal directions and explained variance fractions with the
   original example. Why do uncorrelated variables not necessarily contribute
   equal variance?

</div>
<span style="display: none;">YourTurnEnd</span>

***
## Principal component regression with polynomial features

- Combine PCA with linear regression to get PCR, a supervised ML method
- Components with little predictor variance may still help predict prices
    - CV: choice of retained variance based on prediction accuracy

- Five Ames predictors: `LivingArea`, `LotArea`, `YearBuilt`, `OverallQuality`,
  and `Fireplaces`
    - Quality enters as one continuous covariate on a 1–10 scale
- Same sample restrictions, 70/30 split, and random seed as before

In [ ]:
# Features used for PCR
features_pcr = ['LivingArea', 'LotArea', 'YearBuilt', 'OverallQuality', 'Fireplaces']

# Load the Ames housing data for PCR
X_pcr, y_pcr = load_ames_data(file, features_pcr, target)

# Create the same 70/30 train-test split for PCR
X_pcr_train, X_pcr_test, y_pcr_train, y_pcr_test = train_test_split(
    X_pcr, y_pcr, test_size=0.3, random_state=1234
)

Pipeline for principal component regression with polynomial features:

1. Standardize inputs
2. Polynomial expansion
3. Standardize expanded polynomial features
    - Scaling affects the directions retained by PCA
4. Principal component analysis with
   [`PCA`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)

   - Choose the number of components or share of predictor variance to retain
   - `n_components=0.9`: enough components to explain more than 90% of predictor
     variance
    - `svd_solver='full'`: more stable with highly correlated polynomial columns
5. Linear regression

In [ ]:
from sklearn.decomposition import PCA

# Target fraction of variance to retain in PCA
pca_variance = 0.9

# TODO: Create PCR pipeline for polynomial degree 5
# TODO: Use steps input_scale, poly, scale, pca, and lr
# pipe_pcr = ...

### PCR versus polynomial regression without PCA

- A controlled comparison at polynomial degree 5
- Same five predictors, observations, scaling, folds, and scorer
- The only difference is the PCA step

In [ ]:
# Recreate degree 5 polynomial pipeline without PCA for cross-validation
poly5 = Pipeline(
    steps=[
        ('input_scale', StandardScaler()),
        ('poly', PolynomialFeatures(degree=5, include_bias=False)),
        ('scale', StandardScaler()),
        ('lr', LinearRegression(fit_intercept=True)),
    ]
)

# Compute cross-validation scores for polynomial regression without PCA
poly5_scores = cross_val_score(
    poly5,  # estimator or pipeline
    X_pcr_train,  # training predictors
    y_pcr_train,  # training target
    cv=cv_splitter,  # cross-validation splitter
    scoring=scoring,  # scoring rule
    n_jobs=4,  # number of parallel jobs
)

In [ ]:
# TODO: Compute scores for PCR with cross_val_score()
# pcr5_scores = ...

In [ ]:
# Compare the mean validation RMSEs for both models
df_degree5 = pd.DataFrame(
    {
        'Model': ['Degree 5 polynomial', 'Degree 5 PCR'],
        'Mean validation RMSE': [(-poly5_scores).mean(), (-pcr5_scores).mean()],
    }
)
df_degree5

#### Visualize variance share of each principal component

- Refit on all training observations for inspection
- Extract info about components retained and variance explained from the fitted PCA step:
    - `named_steps['pca']`: fitted PCA step
    - `n_features_in_`: number of expanded predictors
    - `n_components_`: number of retained components

In [ ]:
# Refit the PCR pipeline on the entire training sample
pipe_pcr.fit(X_pcr_train, y_pcr_train)

# TODO: Report n_features_in_ and n_components_ from the fitted PCA step
# pca_step = pipe_pcr.named_steps['pca']

- `explained_variance_ratio_`: variance fraction for each retained component

In [ ]:
# TODO: Extract var_expl from the fitted PCA step

- `plot_pcr_variance()` plots the variance fractions and cumulative fraction
- Bars: individual fractions; line: cumulative fraction
- Dashed line: target fraction

In [ ]:
def plot_pcr_variance(var_expl, n_comp=10, target_var=0.9):
    """
    Plot the explained variance fractions for the retained components in PCR.

    Parameters
    ----------
    var_expl : array of shape (n_components,)
        Explained variance fractions for the retained components
    n_comp : int
        Number of components to plot
    target_var : float
        Target variance fraction to indicate on the plot
    """
    import matplotlib.pyplot as plt
    import numpy as np

    _fig, ax = plt.subplots(figsize=(6, 3.5), layout='constrained')

    # Component number shown on the x-axis
    n_comp = min(n_comp, len(var_expl))
    components = np.arange(1, n_comp + 1)

    bars = ax.bar(components, var_expl[:n_comp], color='steelblue')
    ax.set_ylabel('Fraction of predictor variance')
    ax.set_xlabel('Principal component')
    ax.set_xticks(components)
    ax.set_ylim(0, 1.05)

    # Cumulative explained variance on a second y-axis
    ax_cum = ax.twinx()
    cum_var_expl = np.cumsum(var_expl)
    cumul = ax_cum.plot(components, cum_var_expl[:n_comp], c='black', marker='o', ms=4)
    ax_cum.set_ylabel('Cumulative fraction of predictor variance')
    ax_cum.set_ylim(0, 1.05)
    if target_var is not None:
        ax_cum.axhline(target_var, c='black', ls='--', lw=1)
        ax.text(
            0.5,
            target_var - 0.025,
            f'Target variance fraction: {target_var:.2f}',
            c='black',
            va='top',
            ha='left',
            fontsize=10,
        )

    ax.legend(
        [bars, cumul[0]],
        ['Variance explained', 'Cumulative variance'],
        loc='center right',
    )
    ax.set_title(f'PCR: first {n_comp} retained components')

In [ ]:
# Number of components to plot
n_comp = 15

plot_pcr_variance(var_expl, n_comp=n_comp, target_var=pca_variance)

### Cross-validating multiple hyperparameters

- `GridSearchCV` can optimize over several parameters
    - `poly__degree`: 2, 3, 4, or 5
    - `pca__n_components`: 0.9, 0.95, 0.975, 0.99, or 0.999
- 20 combinations, using the same five-fold CV and negative RMSE
- Same five-predictor training sample for all candidates
- `GridSearchCV` selects and refits the best combination

In [ ]:
# Candidate degrees and variance thresholds (20 combinations)
param_grid = {
    'poly__degree': [2, 3, 4, 5],
    'pca__n_components': [0.9, 0.95, 0.975, 0.99, 0.999],
}

# Define the CV splitter
cv_splitter = KFold(n_splits=5, shuffle=True, random_state=1234)

# Define the scoring metric
scoring = 'neg_root_mean_squared_error'

# TODO: Create pcr_cv with GridSearchCV and refit=True
# pcr_cv = ...

# TODO: Fit pcr_cv on X_pcr_train, y_pcr_train

In [ ]:
# TODO: Report pcr_cv.best_params_ and the CV selection RMSE

- `best_estimator_.named_steps['pca']`: PCA from the final refit
- A fixed variance threshold can retain different counts across folds
- Counts below refer to the final refit on all training observations

In [ ]:
# Extract the PCA step from the best estimator
best_pca = pcr_cv.best_estimator_.named_steps['pca']

# Store number of features entering the model
features_in = best_pca.n_features_in_
# Store number of retained components
n_components = best_pca.n_components_

print(f'Number of expanded polynomial features: {features_in}')
print(f'Number of retained principal components: {n_components}')

### Evaluating the selected models on the test sample

- Rerun polynomial regression CV without PCA using the same five predictors
- Both searches use degrees 2–5, the same folds, and the same scorer
- Each selected model is refitted on all training observations
- Final predictions: same independent test sample

In [ ]:
# Select among degrees 2 to 5 using the same CV setup as PCR
poly_param_grid = {'poly__degree': [2, 3, 4, 5]}

poly_cv = GridSearchCV(
    poly5,  # estimator or pipeline
    poly_param_grid,  # candidate parameter values
    cv=cv_splitter,  # cross-validation splitter
    scoring=scoring,  # scoring rule
    n_jobs=4,  # number of parallel jobs
    refit=True,  # refit on all training observations
)
poly_cv.fit(X_pcr_train, y_pcr_train)

print(poly_cv.best_params_)
print(f'Polynomial CV selection RMSE: {-poly_cv.best_score_:.3f}')

In [ ]:
from sklearn.metrics import r2_score, root_mean_squared_error

# Predict on test sample and compute RMSE and R² for polynomial model without PCA
y_poly_pred = poly_cv.predict(X_pcr_test)
poly_test_rmse = root_mean_squared_error(y_pcr_test, y_poly_pred)
poly_test_r2 = r2_score(y_pcr_test, y_poly_pred)

# Predict on test sample and compute RMSE and R² for PCR model
y_pcr_pred = pcr_cv.predict(X_pcr_test)
pcr_test_rmse = root_mean_squared_error(y_pcr_test, y_pcr_pred)
pcr_test_r2 = r2_score(y_pcr_test, y_pcr_pred)

# Compare the selected degrees, model dimensions, and test scores
poly_features = poly_cv.best_estimator_.named_steps['poly'].n_output_features_
df_comparison = pd.DataFrame(
    {
        'Model': ['Selected polynomial', 'Selected PCR'],
        'Degree': [
            poly_cv.best_params_['poly__degree'],
            pcr_cv.best_params_['poly__degree'],
        ],
        'Regressors': [poly_features, n_components],
        'Test RMSE': [poly_test_rmse, pcr_test_rmse],
        'Test R²': [poly_test_r2, pcr_test_r2],
    }
)
print(df_comparison.to_string(index=False))

              Model  Degree  Regressors  Test RMSE  Test R²
Selected polynomial       3          55  28.588499 0.852468
       Selected PCR       3          40  28.097610 0.857491
